In [11]:
import importlib
import algorithms              # make sure the module itself is imported
importlib.reload(algorithms)   # reloads algorithms.py from disk

from algorithms import spnsa, spnsa_modified, _single_source_shortest_path_basic, calculate_reliance_from_sources, draw_graph


In [12]:
import pandas as pd
import numpy as np
import networkx as nx
import time
from algorithms import spnsa, _single_source_shortest_path_basic, calculate_reliance_from_sources, draw_graph
from helpers import top_nodes_by_degree_imbalance, criminals_per_largest_components


def create_criminal_graph(file_path:str):
    #G = nx.MultiDiGraph()
    G = nx.DiGraph()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            if p[10] == '1':
                v = p[2]
                w = p[4]
                G.add_edge(v, w)

    f.close()
    return G

import csv

def create_graph(file_path: str) -> nx.DiGraph:
    G = nx.DiGraph()
    with open(file_path, newline="") as f:
        reader = csv.reader(f)
        next(reader, None)  # skip header
        for row in reader:
            v = row[2].strip()
            w = row[4].strip()
            G.add_edge(v, w)
    return G

def get_suspicious_nodes(file_path:str):
    suspicious_nodes = set()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            if p[10] == '1':
                suspicious_nodes.add(p[2])
                suspicious_nodes.add(p[4])

    f.close()
    return suspicious_nodes

def get_suspicious_edges(file_path:str):
    suspicious_edges = set()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            if p[10] == '1':
                suspicious_edges.add((p[2], p[4]))

    f.close()
    return suspicious_edges

import networkx as nx


def find_largest_component(
    G: nx.Graph,
    component_type: str = "weak",  # used only if G is directed
) -> nx.Graph:
    """
    Return the largest component as an induced subgraph (copied).

    - If G is undirected: uses connected_components.
    - If G is directed: uses weakly_connected_components by default,
      or strongly_connected_components if component_type="strong".
    """
    if G.number_of_nodes() == 0:
        # preserve graph type
        return G.copy()

    if G.is_directed():
        if component_type not in {"weak", "strong"}:
            raise ValueError("component_type must be 'weak' or 'strong' for directed graphs.")
        comps = (
            nx.weakly_connected_components(G)
            if component_type == "weak"
            else nx.strongly_connected_components(G)
        )
    else:
        comps = nx.connected_components(G)

    largest_cc = max(comps, key=len)

    return G.subgraph(largest_cc).copy()


def get_connected_components(G):
    if nx.number_connected_components(G) == 1:
        return [G]
        
    connected_components_generatorset = nx.connected_components(G)
    connected_components = []
    for component in connected_components_generatorset:
        component_graph = G.subgraph(component).copy()
        connected_components.append(component_graph)
    return connected_components

def get_eccentricity_distribution_list(component_list):
    ''' component_list : List[component] '''
    eccentricity_distribution_list = []
    for component in component_list:
        eccentricity_distribution = nx.eccentricity(component)
        eccentricity_distribution_list.append(eccentricity_distribution)
    return eccentricity_distribution_list




In [14]:
G_all = create_graph('../../datasets/IBM AML/HI-Medium_Trans.csv')
print(G_all)

DiGraph with 2076999 nodes and 4476959 edges


In [15]:
G_criminal = create_criminal_graph('../../datasets/IBM AML/HI-Medium_Trans.csv')
print(G_criminal)

DiGraph with 41857 nodes and 33199 edges


In [16]:
largest_component = find_largest_component(G_all)
print(largest_component)

DiGraph with 1611244 nodes and 4015935 edges


In [19]:
len(largest_component.nodes() & criminal_nodes)

41774

In [17]:
criminal_nodes = get_suspicious_nodes('../../datasets/IBM AML/HI-Medium_Trans.csv')
criminal_nodes_list = list(criminal_nodes)
all_nodes_list = list(largest_component.nodes())
criminal_edges = get_suspicious_edges('../../datasets/IBM AML/HI-Medium_Trans.csv')

In [18]:
# checked that 6342 of the criminals lie in the largets connnected component of the directed G
print(len(criminal_nodes))
print(len(criminal_edges))

41857
33199


##########################################################################################
##########################################################################################
##########################################################################################

In [20]:
import math
import networkx as nx
import random
random.seed(0)
np.random.seed(0)

def motif_scores(
    G: nx.DiGraph,
    candidates,
    max_in=None,
    max_out=None,
    # weights for motifs
    alpha=1.0,   # reciprocity
    beta=0.3,    # relay proxy
    gamma=0.8,   # hub penalty
    # NEW: star-like motifs
    s_in=0.4,    # fan-in star strength
    s_out=0.4,   # fan-out star strength
    s_leaf=0.6,  # star where neighbors are low-degree leaves
    star_ratio=3.0,      # require in/out dominance for "pure" star signal
    leaf_deg_max=2 # neighbor considered "leaf-like" if total degree <= this
):
    """
    Compute motif-based anomaly scores for candidate nodes in a directed transaction graph.

    Motifs/signals used per node v (label-free):
      - recip(v): |In(v) ∩ Out(v)|        (2-cycles / reciprocity)
      - cycle3(v): count of u→v→w with w→u (directed 3-cycles through v)
      - relay(v): |In(v)| * |Out(v)|      (pass-through proxy)
      - star motifs (NEW):
          * fanin_star(v): strong fan-in dominance (many in, few out)
          * fanout_star(v): strong fan-out dominance (many out, few in)
          * leaf_star(v): number of in/out neighbors that are low-degree leaves
      - hub penalty: penalize very large degree nodes (often benign services)

    Score:
      score(v) = log(1+cycle3)
               + alpha * log(1+recip)
               + beta  * log(1+relay)
               + s_in  * log(1+fanin_star)
               + s_out * log(1+fanout_star)
               + s_leaf* log(1+leaf_star)
               - gamma * log(1+deg)

    Parameters
    ----------
    G : nx.DiGraph
        Directed transaction graph.
    candidates : Iterable
        Nodes to score (use a filtered subset for scalability).
    max_in, max_out : int or None
        Optional random downsampling caps for in/out neighborhoods.
    alpha, beta, gamma : float
        Weights for reciprocity / relay / hub-penalty terms.
    s_in, s_out, s_leaf : float
        Weights for star-like motifs (fanin/fanout dominance + leaf-star).
    star_ratio : float
        Minimum dominance ratio to count a node as a fan-in or fan-out star:
          fan-in star if indeg >= star_ratio*(outdeg+1)
          fan-out star if outdeg >= star_ratio*(indeg+1)
    leaf_deg_max : int
        A neighbor u is considered "leaf-like" if total_degree(u) <= leaf_deg_max.

    Returns
    -------
    scores : dict[node, float]
    details : dict[node, dict]  # raw motif counts for debugging
    """

    cand_set = set(candidates)

    # Precompute degrees for leaf detection (use total degree in directed sense)
    total_deg = dict(G.degree())  # in+out for DiGraph

    # Precompute neighborhood sets for candidates
    succ = {v: set(G.successors(v)) for v in candidates}
    pred = {v: set(G.predecessors(v)) for v in candidates}

    scores = {}
    details = {}

    for v in candidates:
        inN = pred[v]
        outN = succ[v]

        # Optional downsampling for very large neighborhoods (stochastic but faster)
        if max_in is not None and len(inN) > max_in:
            #inN = set(random.sample(list(inN), max_in))
            #inN = set(sorted(inN)[:max_in])
            in_list = sorted(inN, key=lambda u: (-total_deg[u], u))[:max_in]
            inN = set(in_list)

        if max_out is not None and len(outN) > max_out:
            #outN = set(random.sample(list(outN), max_out))
            #outN = set(sorted(outN)[:max_out])
            out_list = sorted(outN, key=lambda u: (-total_deg[u], u))[:max_out]
            outN = set(out_list)

        indeg = len(inN)
        outdeg = len(outN)
        deg = indeg + outdeg

        # 1) Reciprocity
        recip = len(inN & outN)

        # 2) Relay proxy
        relay = indeg * outdeg

        # 3) Directed 3-cycles through v: u->v, v->w, w->u
        cycle3 = 0
        # iterate over out-neighbors w and count how many edges w->u land in In(v)
        for w in outN:
            cycle3 += len(set(G.successors(w)) & inN)

        # --- NEW: 4) Star-like motifs ---
        # Fan-in star: many in, few out
        fanin_star = indeg if indeg >= star_ratio * (outdeg + 1) else 0
        # Fan-out star: many out, few in
        fanout_star = outdeg if outdeg >= star_ratio * (indeg + 1) else 0

        # Leaf-star: count low-degree neighbors (more robust than raw star degree)
        leaf_in = sum(1 for u in inN if total_deg.get(u, 0) <= leaf_deg_max)
        leaf_out = sum(1 for u in outN if total_deg.get(u, 0) <= leaf_deg_max)
        leaf_star = leaf_in + leaf_out

        # Final score
        score = (
            math.log1p(cycle3)
            + alpha * math.log1p(recip)
            + beta * math.log1p(relay)
            + s_in * math.log1p(fanin_star)
            + s_out * math.log1p(fanout_star)
            + s_leaf * math.log1p(leaf_star)
            - gamma * math.log1p(deg)
        )

        scores[v] = score
        details[v] = {
            "cycle3": cycle3,
            "recip": recip,
            "relay": relay,
            "deg": deg,
            "fanin_star": fanin_star,
            "fanout_star": fanout_star,
            "leaf_star": leaf_star,
            "leaf_in": leaf_in,
            "leaf_out": leaf_out,
        }

    return scores, details


def coherent_topk_feed(G_lcc: nx.DiGraph, ranked_nodes, k=200, centers=2, d_max=3):
    """
    Select a SPNSA-friendly feed by enforcing locality (coherence) among top-ranked nodes.

    SPNSA performs best when feed nodes are not scattered across the graph, because
    overlapping ego-networks produce more internal connections (more `OC` co-occurrence)
    and prevent shortest-path extraction from drifting through unrelated regions.

    This function converts a ranked list of candidate nodes (e.g., from `motif_scores`)
    into a feed of size k by:
      1) Choosing a small number of high-ranked **anchor/center** nodes.
      2) For each center, collecting nodes within an undirected shortest-path radius
         `d_max` (a local ball around the center).
      3) Filling the feed with the highest-ranked nodes that lie inside these balls.
      4) If still short of k, filling the remainder with the next highest-ranked nodes.

    Parameters
    ----------
    G_lcc : nx.DiGraph
        Graph (ideally the largest connected component) used for distance computation.
        Distances are computed on the **undirected projection** of this graph.
    ranked_nodes : Sequence
        Nodes sorted by decreasing "risk" or "suspicion" score.
    k : int, default=200
        Desired feed size.
    centers : int, default=2
        Number of top-ranked anchors to seed local selection. Increasing `centers`
        allows selecting multiple suspicious regions (useful if illicit activity
        is multi-cluster).
    d_max : int, default=3
        Maximum undirected graph distance from each center to be considered "local".

    Returns
    -------
    feed : list
        List of up to k nodes, prioritized by rank but constrained to be local around
        a few top-ranked centers.

    Notes
    -----
    - Using undirected distances is intentional: for coherence we care about being
      in the same neighborhood regardless of edge direction.
    - Smaller `d_max` typically yields tighter, more coherent feeds (often higher
      illicit ratio), but may reduce coverage of illicit nodes.
    - If ranked_nodes includes nodes not present in G_lcc, they will naturally be
      skipped when they fail the distance-ball membership checks.
    """
    # Use undirected distances for locality (good for “same region”)
    U = G_lcc.to_undirected()

    feed = []
    used = set()

    # pick top `centers` nodes as anchors
    anchor_nodes = ranked_nodes[:centers]
    
    for i, c in enumerate(anchor_nodes):
        if c in used: 
            continue
        # BFS ball within d_max
        dist = nx.single_source_shortest_path_length(U, c, cutoff=d_max)
        ball = set(dist.keys())

        for v in ranked_nodes:
            if v in used:
                continue
            if v in ball:
                feed.append(v)
                used.add(v)
                if len(feed) >= k:
                    return feed

    # if not enough, fill with remaining top-ranked
    for v in ranked_nodes:
        if v not in used:
            feed.append(v)
            if len(feed) >= k:
                break

    return feed

In [21]:
import time
import random
import numpy as np
import pandas as pd
import networkx as nx

def shuffle_edge_insertion_order(G: nx.DiGraph, seed: int) -> nx.DiGraph:
    rng = random.Random(seed)
    H = nx.DiGraph() if G.is_directed() else nx.Graph()
    H.add_nodes_from(list(G.nodes()))
    edges = list(G.edges(data=True))
    rng.shuffle(edges)
    H.add_edges_from(edges)
    return H


def run_once(
    G_base: nx.DiGraph,
    criminal_nodes: set,
    seed: int,
    C: int = 100_000,
    k: int = 200,
    centers: int = 5,
    d_max: int = 2,
    radius: int = 1,
    motif_params=None,
    max_in: int = 500,
    max_out: int = 500,
):
    motif_params = motif_params or dict(alpha=0.2, star_ratio=15.0, s_in=0.3, s_out=0.4)

    # reproducibility
    random.seed(seed)
    np.random.seed(seed)

    t0 = time.perf_counter()

    # perturb adjacency order deterministically
    G = shuffle_edge_insertion_order(G_base, seed=seed)

    # ---- stage 1: proxy + candidates ----
    t1 = time.perf_counter()
    in_deg  = dict(G.in_degree())
    out_deg = dict(G.out_degree())
    proxy = {v: in_deg[v] * out_deg[v] for v in G.nodes()}
    candidates = sorted(G.nodes(), key=lambda v: (-proxy[v], str(v)))[:C]
    t2 = time.perf_counter()

    # ---- stage 2: motif scoring ----
    scores, details = motif_scores(
        G, candidates,
        max_in=max_in, max_out=max_out,
        **motif_params
    )
    ranked = sorted(scores.keys(), key=lambda v: (-scores[v], str(v)))
    t3 = time.perf_counter()

    # ---- stage 3: feed selection ----
    feed = coherent_topk_feed(G, ranked, k=k, centers=centers, d_max=d_max)
    t4 = time.perf_counter()

    # ---- stage 4: SPNSA ----
    SPN, paths = spnsa(G, feed, radius=radius)
    t5 = time.perf_counter()

    # number of connected components in the undirected projection
    n_components = nx.number_connected_components(SPN.to_undirected())

    U = SPN.to_undirected()
    cc_sizes = [len(c) for c in nx.connected_components(U)]
    lcc_frac = (max(cc_sizes) / SPN.number_of_nodes()) if cc_sizes else 0.0


    # metrics
    spn_nodes = set(SPN.nodes())
    illicit = len(spn_nodes & criminal_nodes)
    ratio = illicit / max(1, SPN.number_of_nodes())

    return {
        "seed": seed,

        # sizes
        "cand_size": len(candidates),
        "feed_size": len(feed),
        "spn_nodes": SPN.number_of_nodes(),
        "spn_edges": SPN.number_of_edges(),
        "spn_components": n_components,
        "spn_lcc_frac": lcc_frac, # fraction of largest component order to |V_spn|
        "illicit_nodes": illicit,
        "illicit_ratio": ratio,

        # timing (seconds)
        "t_total": t5 - t1,
        "t_candidates": t2 - t1,
        "t_motif": t3 - t2,
        "t_feed": t4 - t3,
        "t_spnsa": t5 - t4,
    }


def summarize_series(x: np.ndarray):
    x = np.asarray(x, dtype=float)
    return {
        "mean": float(x.mean()),
        "std": float(x.std(ddof=1)) if len(x) > 1 else 0.0,
        "median": float(np.median(x)),
        "q25": float(np.quantile(x, 0.25)),
        "q75": float(np.quantile(x, 0.75)),
        "min": float(x.min()),
        "max": float(x.max()),
    }


def run_repeated_experiment(
    G: nx.DiGraph,
    criminal_nodes: set,
    R: int = 20,
    seeds=None,
    **kwargs
):
    if seeds is None:
        seeds = list(range(R))

    rows = []
    for s in seeds:
        rows.append(run_once(G, criminal_nodes, seed=s, **kwargs))

    df = pd.DataFrame(rows)

    # ---- print summary for illicit ratio ----
    ratio_sum = summarize_series(df["illicit_ratio"].to_numpy())
    print("Illicit ratio over runs:")
    print(f"  mean ± std : {ratio_sum['mean']:.4f} ± {ratio_sum['std']:.4f}")
    print(f"  median [IQR]: {ratio_sum['median']:.4f} [{ratio_sum['q25']:.4f}, {ratio_sum['q75']:.4f}]")
    print(f"  min..max   : {ratio_sum['min']:.4f} .. {ratio_sum['max']:.4f}")

    # ---- print summary for connected components ----
    print("\nConnected components in SPN (undirected) over runs:")
    cc = summarize_series(df["spn_components"].to_numpy())
    print(f"  mean ± std : {cc['mean']:.2f} ± {cc['std']:.2f}")
    print(f"  median [IQR]: {cc['median']:.0f} [{cc['q25']:.0f}, {cc['q75']:.0f}]")
    print(f"  min..max   : {cc['min']:.0f} .. {cc['max']:.0f}")

    # ---- print summary for spn nodes ----
    print("\nNumber of nodes in SPN over runs:")
    sn = summarize_series(df["spn_nodes"].to_numpy())
    print(f"  mean ± std : {sn['mean']:.2f} ± {sn['std']:.2f}")
    print(f"  median [IQR]: {sn['median']:.0f} [{sn['q25']:.0f}, {sn['q75']:.0f}]")
    print(f"  min..max   : {sn['min']:.0f} .. {sn['max']:.0f}")

    # ---- print summary for spn edges ----
    print("\nNumber of edges in SPN over runs:")
    se = summarize_series(df["spn_edges"].to_numpy())
    print(f"  mean ± std : {se['mean']:.2f} ± {se['std']:.2f}")
    print(f"  median [IQR]: {se['median']:.0f} [{se['q25']:.0f}, {se['q75']:.0f}]")
    print(f"  min..max   : {se['min']:.0f} .. {se['max']:.0f}")

    # ---- print summary for spn lcc ----
    print("\nFraction of lcc in SPN over runs:")
    slcc = summarize_series(df["spn_lcc_frac"].to_numpy())
    print(f"  mean ± std : {slcc['mean']:.2f} ± {slcc['std']:.2f}")
    print(f"  median [IQR]: {slcc['median']:.0f} [{slcc['q25']:.0f}, {slcc['q75']:.0f}]")
    print(f"  min..max   : {slcc['min']:.0f} .. {slcc['max']:.0f}")

    # ---- print summary for spn illicit nodes ----
    print("\nNumber of illicit nodes in SPN over runs:")
    ssn = summarize_series(df["illicit_nodes"].to_numpy())
    print(f"  mean ± std : {ssn['mean']:.2f} ± {ssn['std']:.2f}")
    print(f"  median [IQR]: {ssn['median']:.0f} [{ssn['q25']:.0f}, {ssn['q75']:.0f}]")
    print(f"  min..max   : {ssn['min']:.0f} .. {ssn['max']:.0f}")
    
    # ---- print timing summaries ----
    print("\nRuntime (seconds) over runs:")
    for col, name in [
        ("t_total", "Total"),
        ("t_candidates", "Candidates"),
        ("t_motif", "Motif scoring"),
        ("t_feed", "Feed selection"),
        ("t_spnsa", "SPNSA"),
    ]:
        s = summarize_series(df[col].to_numpy())
        print(f"  {name:14s}: mean±std {s['mean']:.3f}±{s['std']:.3f} | "
              f"median[IQR] {s['median']:.3f}[{s['q25']:.3f},{s['q75']:.3f}] | "
              f"min..max {s['min']:.3f}..{s['max']:.3f}")

    return df, {"illicit_ratio": ratio_sum,
                "t_total": summarize_series(df["t_total"].to_numpy())}



In [23]:
# feed size 200; radius 1
motif_kwargs = dict(alpha=0.2, star_ratio=15.0, s_in=0.3, s_out=0.4)

df_runs, summary = run_repeated_experiment(
    G=largest_component,
    criminal_nodes=criminal_nodes,
    R=2,
    C=300000,
    k=200,
    centers=5,
    d_max=2,
    radius=1,
    motif_params=motif_kwargs,
    max_in=500,
    max_out=500,
)

display(df_runs.sort_values("illicit_ratio", ascending=False).head(20))


Illicit ratio over runs:
  mean ± std : 0.2659 ± 0.0024
  median [IQR]: 0.2659 [0.2650, 0.2667]
  min..max   : 0.2642 .. 0.2676

Connected components in SPN (undirected) over runs:
  mean ± std : 15.00 ± 0.00
  median [IQR]: 15 [15, 15]
  min..max   : 15 .. 15

Number of nodes in SPN over runs:
  mean ± std : 370.50 ± 0.71
  median [IQR]: 370 [370, 371]
  min..max   : 370 .. 371

Number of edges in SPN over runs:
  mean ± std : 907.50 ± 0.71
  median [IQR]: 908 [907, 908]
  min..max   : 907 .. 908

Fraction of lcc in SPN over runs:
  mean ± std : 0.80 ± 0.00
  median [IQR]: 1 [1, 1]
  min..max   : 1 .. 1

Number of illicit nodes in SPN over runs:
  mean ± std : 98.50 ± 0.71
  median [IQR]: 98 [98, 99]
  min..max   : 98 .. 99

Runtime (seconds) over runs:
  Total         : mean±std 4841.557±949.816 | median[IQR] 4841.557[4505.747,5177.368] | min..max 4169.936..5513.179
  Candidates    : mean±std 2.458±0.072 | median[IQR] 2.458[2.433,2.484] | min..max 2.408..2.509
  Motif scoring : mean±

,seed,cand_size,feed_size,spn_nodes,spn_edges,spn_components,spn_lcc_frac,illicit_nodes,illicit_ratio,t_total,t_candidates,t_motif,t_feed,t_spnsa
0,0,300000,200,370,907,15,0.800000,99,0.267568,4169.935874,2.509129,3.705405,12.374859,4151.346481
1,1,300000,200,371,908,15,0.800539,98,0.264151,5513.178996,2.407810,3.945810,7.313907,5499.511470


In [24]:
# feed size 200; radius 2
motif_kwargs = dict(alpha=0.2, star_ratio=15.0, s_in=0.3, s_out=0.4)

df_runs, summary = run_repeated_experiment(
    G=largest_component,
    criminal_nodes=criminal_nodes,
    R=2,
    C=300000,
    k=200,
    centers=5,
    d_max=2,
    radius=2,
    motif_params=motif_kwargs,
    max_in=500,
    max_out=500,
)

display(df_runs.sort_values("illicit_ratio", ascending=False).head(20))


Illicit ratio over runs:
  mean ± std : 0.2861 ± 0.0005
  median [IQR]: 0.2861 [0.2859, 0.2863]
  min..max   : 0.2857 .. 0.2865

Connected components in SPN (undirected) over runs:
  mean ± std : 12.00 ± 0.00
  median [IQR]: 12 [12, 12]
  min..max   : 12 .. 12

Number of nodes in SPN over runs:
  mean ± std : 377.50 ± 0.71
  median [IQR]: 378 [377, 378]
  min..max   : 377 .. 378

Number of edges in SPN over runs:
  mean ± std : 973.50 ± 0.71
  median [IQR]: 974 [973, 974]
  min..max   : 973 .. 974

Fraction of lcc in SPN over runs:
  mean ± std : 0.87 ± 0.00
  median [IQR]: 1 [1, 1]
  min..max   : 1 .. 1

Number of illicit nodes in SPN over runs:
  mean ± std : 108.00 ± 0.00
  median [IQR]: 108 [108, 108]
  min..max   : 108 .. 108

Runtime (seconds) over runs:
  Total         : mean±std 40316.589±1700.597 | median[IQR] 40316.589[39715.338,40917.841] | min..max 39114.086..41519.093
  Candidates    : mean±std 2.787±0.195 | median[IQR] 2.787[2.718,2.856] | min..max 2.649..2.925
  Motif sc

,seed,cand_size,feed_size,spn_nodes,spn_edges,spn_components,spn_lcc_frac,illicit_nodes,illicit_ratio,t_total,t_candidates,t_motif,t_feed,t_spnsa
1,1,300000,200,377,973,12,0.870027,108,0.286472,39114.086104,2.925324,4.306513,18.419536,39088.43473
0,0,300000,200,378,974,12,0.870370,108,0.285714,41519.092853,2.649078,3.963141,17.006455,41495.47418


In [25]:
# feed size 500; radius 1
motif_kwargs = dict(alpha=0.2, star_ratio=15.0, s_in=0.3, s_out=0.4)

df_runs, summary = run_repeated_experiment(
    G=largest_component,
    criminal_nodes=criminal_nodes,
    R=2,
    C=300000,
    k=500,
    centers=5,
    d_max=2,
    radius=1,
    motif_params=motif_kwargs,
    max_in=500,
    max_out=500,
)

display(df_runs.sort_values("illicit_ratio", ascending=False).head(20))


Illicit ratio over runs:
  mean ± std : 0.2065 ± 0.0010
  median [IQR]: 0.2065 [0.2061, 0.2069]
  min..max   : 0.2058 .. 0.2072

Connected components in SPN (undirected) over runs:
  mean ± std : 190.00 ± 0.00
  median [IQR]: 190 [190, 190]
  min..max   : 190 .. 190

Number of nodes in SPN over runs:
  mean ± std : 830.50 ± 0.71
  median [IQR]: 830 [830, 831]
  min..max   : 830 .. 831

Number of edges in SPN over runs:
  mean ± std : 1623.50 ± 0.71
  median [IQR]: 1624 [1623, 1624]
  min..max   : 1623 .. 1624

Fraction of lcc in SPN over runs:
  mean ± std : 0.38 ± 0.00
  median [IQR]: 0 [0, 0]
  min..max   : 0 .. 0

Number of illicit nodes in SPN over runs:
  mean ± std : 171.50 ± 0.71
  median [IQR]: 172 [171, 172]
  min..max   : 171 .. 172

Runtime (seconds) over runs:
  Total         : mean±std 8056.399±912.174 | median[IQR] 8056.399[7733.897,8378.901] | min..max 7411.395..8701.404
  Candidates    : mean±std 2.603±0.012 | median[IQR] 2.603[2.598,2.607] | min..max 2.594..2.611
  Mot

,seed,cand_size,feed_size,spn_nodes,spn_edges,spn_components,spn_lcc_frac,illicit_nodes,illicit_ratio,t_total,t_candidates,t_motif,t_feed,t_spnsa
0,0,300000,500,830,1623,190,0.375904,172,0.207229,8701.403708,2.611340,3.947857,18.202746,8676.641765
1,1,300000,500,831,1624,190,0.376655,171,0.205776,7411.394634,2.594052,3.960610,17.812080,7387.027892


##########################################################################################
##########################################################################################
##########################################################################################

In [ ]:
# Cell 1 — Imports
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import random
import os

# Cell 2 — Plotting utilities

def _safe_undirected_layout(G, layout="spring", seed=42, **kwargs):
    """
    Compute 2D positions for plotting. We compute layout on an undirected projection
    for stability, then reuse for drawing the directed graph.
    """
    U = G.to_undirected() if G.is_directed() else G

    if layout == "spring":
        return nx.spring_layout(U, seed=seed, **kwargs)
    if layout == "kamada":
        return nx.kamada_kawai_layout(U, **kwargs)
    if layout == "shell":
        return nx.shell_layout(U)
    if layout == "circular":
        return nx.circular_layout(U)

    # default fallback
    return nx.spring_layout(U, seed=seed, **kwargs)


def save_spn_components(
    SPN: nx.Graph,
    illicit_nodes=None,
    illicit_edges=None,
    save_dir="spn_components",
    prefix="spn_comp",
    max_components=50,     # save first N comps by size
    min_size=5,            # skip tiny comps
    layout="spring",
    seed=42,
    figsize=(10, 8),
    node_size=35,
    alpha_edges=0.35,
    arrows=False,
    dpi=200,
    save_pdf=False
):
    """
    Save each (weakly) connected component of SPN as a separate image file.

    Files created:
      save_dir/prefix_001_nodesXXX_edgesYYY.png  (and .pdf if save_pdf=True)
    """
    os.makedirs(save_dir, exist_ok=True)

    illicit_nodes = set() if illicit_nodes is None else set(illicit_nodes)
    illicit_edges = set() if illicit_edges is None else set(illicit_edges)

    # Components
    if SPN.is_directed():
        comps = list(nx.weakly_connected_components(SPN))
    else:
        comps = list(nx.connected_components(SPN))
    comps = sorted(comps, key=len, reverse=True)

    saved_paths = []
    saved = 0

    for comp_nodes in comps:
        if len(comp_nodes) < min_size:
            continue
        if saved >= max_components:
            break

        H = SPN.subgraph(comp_nodes).copy()

        # Layout computed on undirected projection for stability
        U = H.to_undirected() if H.is_directed() else H
        if layout == "spring":
            pos = nx.spring_layout(U, seed=seed, iterations=150)
        elif layout == "kamada":
            pos = nx.kamada_kawai_layout(U)
        elif layout == "circular":
            pos = nx.circular_layout(U)
        else:
            pos = nx.spring_layout(U, seed=seed, iterations=150)

        nodes = list(H.nodes())
        node_colors = ["tab:red" if n in illicit_nodes else "lightgray" for n in nodes]

        edges = list(H.edges())
        edge_colors, edge_widths = [], []
        for (u, v) in edges:
            illicit = (u, v) in illicit_edges or (v, u) in illicit_edges
            edge_colors.append("tab:red" if illicit else "gray")
            edge_widths.append(1.2 if illicit else 0.6)

        comp_illicit = sum(1 for n in nodes if n in illicit_nodes)
        comp_ratio = comp_illicit / len(nodes)

        plt.figure(figsize=figsize)
        plt.axis("off")
        plt.title(f"Component {saved+1} (|V|={H.number_of_nodes()}, |E|={H.number_of_edges()})")

        nx.draw_networkx_edges(
            H, pos,
            edge_color=edge_colors,
            width=edge_widths,
            alpha=alpha_edges,
            arrows=arrows,
            arrowstyle="-|>" if arrows else None,
            arrowsize=10 if arrows else None,
            connectionstyle="arc3,rad=0.05" if arrows else "arc3,rad=0.0",
        )
        nx.draw_networkx_nodes(
            H, pos,
            node_color=node_colors,
            node_size=node_size,
            linewidths=0.0
        )

        plt.text(
            0.01, 0.01,
            f"illicit nodes: {comp_illicit} | illicit ratio: {comp_ratio:.3f}",
            transform=plt.gca().transAxes
        )

        # File names
        fname = f"{prefix}_{saved+1:03d}_nodes{H.number_of_nodes()}_edges{H.number_of_edges()}"
        png_path = os.path.join(save_dir, fname + ".png")
        plt.savefig(png_path, dpi=dpi, bbox_inches="tight")
        saved_paths.append(png_path)

        if save_pdf:
            pdf_path = os.path.join(save_dir, fname + ".pdf")
            plt.savefig(pdf_path, bbox_inches="tight")
            saved_paths.append(pdf_path)

        plt.close()
        saved += 1

    print(f"Saved {saved} component figure(s) to: {os.path.abspath(save_dir)}")
    return saved_paths



In [ ]:

# Assuming you already have:
# SPN, paths = spnsa(G_all_or_lcc, feed, radius=1)
# criminal_nodes = set(...)
# criminal_edges = set(...)

paths = save_spn_components(
    SPN,
    illicit_nodes=criminal_nodes,
    illicit_edges=criminal_edges,
    save_dir="spn_components_motif_based_feed200_pool300k_radi1_ratio0.18",
    prefix="motig_based_feed200",
    max_components=30,
    min_size=5,
    layout="spring",
    arrows=True,
    save_pdf=False
)